In [ ]:
# Cell 1: Setup Paths
from pathlib import Path

# Detect project root (parent of notebooks/)
ROOT = Path.cwd().parents[0]

# Define paths used in the pipeline
DATA_TRAIN   = ROOT / "data" / "processed" / "training_fe_full.csv"
DATA_LABELS  = ROOT / "data" / "raw" / "training_set_labels.csv"
DATA_TEST    = ROOT / "data" / "processed" / "test_fe_full.csv"

ARTIFACTS    = ROOT / "artifacts_lgbm"
ARTIFACTS.mkdir(exist_ok=True, parents=True)

print("ROOT:       ", ROOT)
print("DATA_TRAIN: ", DATA_TRAIN)
print("DATA_LABELS:", DATA_LABELS)
print("DATA_TEST:  ", DATA_TEST)
print("ARTIFACTS:  ", ARTIFACTS)

In [ ]:

# Cell 2: Add Project Root to Path
import sys
import os

# Add root to path if not present
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
    
print(f"✓ Added {ROOT} to Python path")

# Cell 3: Train Baseline Models
from src.modeling_lgbm import train_baseline_models

print("Training baseline models...")
baseline_results = train_baseline_models(
    train_features=str(DATA_TRAIN),
    train_labels=str(DATA_LABELS),
    outdir=str(ARTIFACTS)
)

print("\n=== Baseline Results ===")
for target, metrics in baseline_results.items():
    print(f"\n{target}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

# Cell 4: Load Training Data for Fine-Tuning
from src.modeling_lgbm import load_training_data, TARGET_COLS

print("Loading training data...")
X, y = load_training_data(str(DATA_TRAIN), str(DATA_LABELS))

print(f"✓ Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"✓ Targets: {TARGET_COLS}")

# Cell 5: Fine-Tune H1N1 Model
from src.modeling_lgbm import tune_model

print("Fine-tuning h1n1_vaccine model...")
h1n1_params, h1n1_metrics = tune_model(
    X, 
    y["h1n1_vaccine"], 
    "h1n1_vaccine", 
    outdir=str(ARTIFACTS)
)

print("\n=== H1N1 Tuned Model ===")
print("Best Parameters:", h1n1_params)
print("\nMetrics:")
for metric, value in h1n1_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Cell 6: Fine-Tune Seasonal Flu Model
print("Fine-tuning seasonal_vaccine model...")
seasonal_params, seasonal_metrics = tune_model(
    X, 
    y["seasonal_vaccine"], 
    "seasonal_vaccine", 
    outdir=str(ARTIFACTS)
)

print("\n=== Seasonal Vaccine Tuned Model ===")
print("Best Parameters:", seasonal_params)
print("\nMetrics:")
for metric, value in seasonal_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Cell 7: Store Tuning Results
tuned_params = {
    "h1n1_vaccine": h1n1_params,
    "seasonal_vaccine": seasonal_params
}

tuned_metrics = {
    "h1n1_vaccine": h1n1_metrics,
    "seasonal_vaccine": seasonal_metrics
}

print("✓ Tuning completed for both targets")

# Cell 8: Train Final H1N1 Model on Full Dataset
from src.modeling_lgbm import train_final_model

print("Training final h1n1_vaccine model on full dataset...")
final_h1n1 = train_final_model(
    X, 
    y["h1n1_vaccine"], 
    "h1n1_vaccine", 
    tuned_params["h1n1_vaccine"],
    outdir=str(ARTIFACTS)
)

# Cell 9: Train Final Seasonal Flu Model on Full Dataset
print("Training final seasonal_vaccine model on full dataset...")
final_seasonal = train_final_model(
    X, 
    y["seasonal_vaccine"], 
    "seasonal_vaccine", 
    tuned_params["seasonal_vaccine"],
    outdir=str(ARTIFACTS)
)

# Cell 10: Load Final Models for Prediction
from src.modeling_lgbm import load_final_models

print("Loading final models for prediction...")
final_models = load_final_models(str(ARTIFACTS))

print(f"✓ Loaded {len(final_models)} final models")
for target in TARGET_COLS:
    print(f"  - {target}")

# Cell 11: Generate Test Predictions
from src.modeling_lgbm import predict_test

print("Generating test predictions...")
df_submit = predict_test(final_models, str(DATA_TEST))

print(f"✓ Generated predictions for {len(df_submit)} test samples")
print("\nSubmission Preview:")
print(df_submit.head(10))

# Cell 12: Save Submission File
submission_path = ROOT / "submission.csv"
df_submit.to_csv(submission_path, index=False)

print(f"✓ Submission saved to: {submission_path}")
print(f"  Shape: {df_submit.shape}")
print(f"  Columns: {list(df_submit.columns)}")

# Cell 13: Summary Statistics
print("\n=== SUBMISSION SUMMARY ===")
print(f"\nTotal predictions: {len(df_submit)}")
print("\nH1N1 Vaccine Probabilities:")
print(df_submit["h1n1_vaccine"].describe())
print("\nSeasonal Vaccine Probabilities:")
print(df_submit["seasonal_vaccine"].describe())

# Cell 14: Compare Baseline vs Tuned Performance
import json

print("\n=== PERFORMANCE COMPARISON ===")

for target in TARGET_COLS:
    print(f"\n{target.upper().replace('_', ' ')}:")
    print("-" * 50)
    
    # Load baseline metrics
    with open(ARTIFACTS / f"baseline_{target}_metrics.json") as f:
        baseline = json.load(f)
    
    # Load tuned metrics
    with open(ARTIFACTS / f"tuned_{target}_metrics.json") as f:
        tuned = json.load(f)
    
    print(f"{'Metric':<15} {'Baseline':<12} {'Tuned':<12} {'Change':<12}")
    print("-" * 50)
    
    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        base_val = baseline[metric]
        tune_val = tuned[metric]
        change = tune_val - base_val
        change_pct = (change / base_val * 100) if base_val > 0 else 0
        
        print(f"{metric:<15} {base_val:<12.4f} {tune_val:<12.4f} {change:+.4f} ({change_pct:+.2f}%)")